# Week 2 - SQL Exercises: Window Functions

## Practice Problems - ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD, Running Totals, Top-N Per Group, CTEs + Window Functions

This notebook contains 7 problems of increasing difficulty. Each problem asks you to write a SQL query using **window functions** against the `company` schema in your PostgreSQL database.

### How to use this notebook:
1. Make sure your PostgreSQL database is running (Docker container with `week2_db`).
2. Run the connection helper cell below to enable `run_query()`.
3. Write your SQL query in each empty code cell.
4. If you get stuck, scroll down to the **HINT** cell (but try first!).
5. After attempting the problem, scroll further to the **SOLUTION** to check your answer.

### Schema reference:
All tables are in the `company` schema. Key tables for these exercises:
- **departments**: `dept_id, dept_name, location, budget`
- **employees**: `emp_id, first_name, last_name, email, department_id, salary, hire_date, manager_id, is_active`
- **products**: `product_id, product_name, category, unit_price`
- **sales**: `sale_id, employee_id, product_id, quantity, sale_date, region`

### Quick window function refresher:
- `ROW_NUMBER() OVER (PARTITION BY x ORDER BY y)` - unique sequential numbers within each partition
- `RANK() OVER (PARTITION BY x ORDER BY y)` - ranking with gaps after ties (1, 2, 2, 4)
- `DENSE_RANK() OVER (PARTITION BY x ORDER BY y)` - ranking without gaps (1, 2, 2, 3)
- `LAG(col, n) OVER (ORDER BY x)` - value from n rows before
- `LEAD(col, n) OVER (ORDER BY x)` - value from n rows after
- `SUM(col) OVER (ORDER BY x)` - running/cumulative total
- `AVG(col) OVER (PARTITION BY x)` - aggregate computed per partition, not collapsing rows

---
### Connection Helper
Run this cell once to set up the `run_query()` helper function.

In [ ]:
import psycopg2
import pandas as pd
from IPython.display import display, HTML

def run_query(query):
    """Execute a SQL query against the week2_db PostgreSQL database and return a DataFrame."""
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="week2_db",
        user="student",
        password="student123"
    )
    try:
        df = pd.read_sql_query(query, conn)
        display(df)
        return df
    except Exception as e:
        print(f"Error: {e}")
        return None
    finally:
        conn.close()

---

## Problem 1: Ranking Employees by Salary Within Each Department

**Business Question:**  
HR is reviewing compensation fairness. For every employee, show their first name, last name, department name, and salary. Then add **three ranking columns** that rank employees by salary (highest to lowest) within each department:
- `row_num` using `ROW_NUMBER()`
- `rank` using `RANK()`
- `dense_rank` using `DENSE_RANK()`

This will let you see the difference between the three functions when employees have **tied salaries** (for example, David Kim and Eva Patel both earn 88000 in Engineering, and Quinn Walker and Rachel Hall both earn 78000 in Finance). Order the results by department name, then by salary descending.

In [ ]:
# Write your query here
query_1 = """

"""
run_query(query_1)

> **HINT** - scroll down only if stuck  
> Join `employees` to `departments` to get the department name. Then use three window functions, each with `OVER (PARTITION BY d.dept_name ORDER BY e.salary DESC)`. The key difference: `ROW_NUMBER()` always gives unique values, `RANK()` skips numbers after ties (1, 2, 2, 4), and `DENSE_RANK()` does not skip (1, 2, 2, 3).

<details>
<summary><strong>SOLUTION - Problem 1</strong> (click to reveal)</summary>

```sql
SELECT e.first_name,
       e.last_name,
       d.dept_name,
       e.salary,
       ROW_NUMBER() OVER (PARTITION BY d.dept_name ORDER BY e.salary DESC) AS row_num,
       RANK()       OVER (PARTITION BY d.dept_name ORDER BY e.salary DESC) AS rank,
       DENSE_RANK() OVER (PARTITION BY d.dept_name ORDER BY e.salary DESC) AS dense_rank
FROM company.employees e
JOIN company.departments d
  ON e.department_id = d.dept_id
ORDER BY d.dept_name, e.salary DESC;
```

**Explanation:** All three window functions share the same `OVER` clause: partitioning by department and ordering by salary descending. The difference appears when two employees in the same department have the same salary:

| Employee | Salary | ROW_NUMBER | RANK | DENSE_RANK |
|----------|--------|------------|------|------------|
| David K. | 88000  | 4          | 4    | 4          |
| Eva P.   | 88000  | 5          | 4    | 4          |
| Frank W. | 75000  | 6          | 6    | 5          |

- **ROW_NUMBER**: Gives each row a unique number (4 and 5), but the assignment between tied rows is arbitrary.
- **RANK**: Gives both tied employees rank 4, then skips to 6 for the next employee.
- **DENSE_RANK**: Gives both tied employees rank 4, then continues with 5 (no gaps).

**When to use which:**
- `ROW_NUMBER`: When you need exactly one row per position (e.g., Top-N queries).
- `RANK`: When ties should share the same rank and you want to see how many people are ahead.
- `DENSE_RANK`: When ties should share the same rank but you want consecutive numbering.
</details>

---

## Problem 2: Top 3 Highest-Paid Employees Per Department (Top-N Per Group)

**Business Question:**  
Management wants to see the **3 highest-paid employees** in **each department**. Show the department name, employee first name, last name, and salary. This is the classic **Top-N per group** pattern, which is best solved using `ROW_NUMBER()` inside a CTE, then filtering the outer query.

Order the final results by department name, then by salary descending.

In [ ]:
# Write your query here
query_2 = """

"""
run_query(query_2)

> **HINT** - scroll down only if stuck  
> This is the classic Top-N per group pattern. Step 1: Write a CTE that joins employees to departments and adds `ROW_NUMBER() OVER (PARTITION BY d.dept_name ORDER BY e.salary DESC) AS rn`. Step 2: In the outer query, `SELECT * FROM cte WHERE rn <= 3`. Note that `ROW_NUMBER()` picks arbitrarily which employee to keep if there are ties. If you want to include all tied employees at the cutoff, use `RANK()` instead.

<details>
<summary><strong>SOLUTION - Problem 2</strong> (click to reveal)</summary>

```sql
WITH ranked AS (
    SELECT e.first_name,
           e.last_name,
           d.dept_name,
           e.salary,
           ROW_NUMBER() OVER (PARTITION BY d.dept_name ORDER BY e.salary DESC) AS rn
    FROM company.employees e
    JOIN company.departments d
      ON e.department_id = d.dept_id
)
SELECT dept_name,
       first_name,
       last_name,
       salary
FROM ranked
WHERE rn <= 3
ORDER BY dept_name, salary DESC;
```

**Explanation:** This is the most important pattern to master for window function interviews and real-world work:

1. **CTE (`ranked`)**: Assigns a row number within each department, ordered by salary descending. The highest-paid person in each department gets `rn = 1`, second gets `rn = 2`, etc.
2. **Outer query**: Filters to `rn <= 3`, keeping only the top 3 per department.

**Why a CTE?** Window functions cannot appear in a `WHERE` clause. They are evaluated after `WHERE` filtering, so you must first compute them in a subquery or CTE, then filter in an outer query.

**Tie-breaking behavior:** `ROW_NUMBER()` arbitrarily assigns different row numbers to employees with the same salary. In Engineering, both David Kim and Eva Patel earn 88000; one gets `rn = 4` and the other `rn = 5`, so only one might appear in the top 3 depending on internal ordering. To include all ties, swap `ROW_NUMBER()` for `RANK()`.

**This pattern works for any Top-N:** Change `rn <= 3` to `rn <= 5` for top 5, `rn <= 1` for the single highest per group, etc.
</details>

---

## Problem 3: Days Between Consecutive Sales in the Same Region (LAG)

**Business Question:**  
The Sales operations team wants to understand sale frequency. For each sale, show the `sale_id`, `region`, `sale_date`, and the **number of days since the previous sale in the same region**. Use `LAG()` to get the previous sale date within each region, then calculate the difference between the current date and the previous date.

For the first sale in each region (where there is no previous sale), the days difference should be NULL. Order by region, then by sale_date.

In [ ]:
# Write your query here
query_3 = """

"""
run_query(query_3)

> **HINT** - scroll down only if stuck  
> Use `LAG(sale_date) OVER (PARTITION BY region ORDER BY sale_date)` to get the previous sale date within the same region. Then subtract: `sale_date - prev_sale_date` gives the difference in days. Filter out rows where `region IS NULL` (or handle them separately), since NULL regions cannot be meaningfully partitioned.

<details>
<summary><strong>SOLUTION - Problem 3</strong> (click to reveal)</summary>

```sql
SELECT sale_id,
       region,
       sale_date,
       LAG(sale_date) OVER (PARTITION BY region ORDER BY sale_date) AS prev_sale_date,
       sale_date - LAG(sale_date) OVER (PARTITION BY region ORDER BY sale_date) AS days_since_prev_sale
FROM company.sales
WHERE region IS NOT NULL
ORDER BY region, sale_date;
```

**Explanation:**

- `LAG(sale_date) OVER (PARTITION BY region ORDER BY sale_date)` looks back one row within each region, ordered chronologically. For the first sale in each region, there is no previous row, so `prev_sale_date` is NULL.
- `sale_date - prev_sale_date` computes the difference in days. When `prev_sale_date` is NULL, the result is also NULL, which is the correct behavior for the first sale.
- `WHERE region IS NOT NULL` excludes sales with no region assigned, since partitioning by NULL would group all NULL-region sales together, which is not meaningful for this analysis.

**Alternative (cleaner) approach using a CTE to avoid calling LAG twice:**

```sql
WITH sale_with_prev AS (
    SELECT sale_id,
           region,
           sale_date,
           LAG(sale_date) OVER (PARTITION BY region ORDER BY sale_date) AS prev_sale_date
    FROM company.sales
    WHERE region IS NOT NULL
)
SELECT sale_id,
       region,
       sale_date,
       prev_sale_date,
       sale_date - prev_sale_date AS days_since_prev_sale
FROM sale_with_prev
ORDER BY region, sale_date;
```

**Real-world use:** This pattern is essential for calculating time between events (days between purchases, hours between support tickets, etc.).
</details>

---

## Problem 4: Running Total of Sales Revenue

**Business Question:**  
Finance needs to track cumulative sales revenue over time. For each sale, calculate the **revenue** (`quantity * unit_price`) and compute a **running total** of revenue ordered by `sale_date`. Show the `sale_id`, `sale_date`, `quantity`, `unit_price`, the line `revenue`, and the `running_total`. Order by `sale_date`, then `sale_id`.

Join `sales` with `products` to get the unit price.

In [ ]:
# Write your query here
query_4 = """

"""
run_query(query_4)

> **HINT** - scroll down only if stuck  
> Join `sales` to `products` on `product_id`. Calculate revenue as `quantity * unit_price`. For the running total, use `SUM(quantity * unit_price) OVER (ORDER BY sale_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`. The `ROWS BETWEEN` clause makes the cumulative sum explicit (though in PostgreSQL, `SUM(...) OVER (ORDER BY ...)` defaults to this behavior).

<details>
<summary><strong>SOLUTION - Problem 4</strong> (click to reveal)</summary>

```sql
SELECT s.sale_id,
       s.sale_date,
       s.quantity,
       p.unit_price,
       ROUND(s.quantity * p.unit_price, 2) AS revenue,
       ROUND(SUM(s.quantity * p.unit_price) OVER (
           ORDER BY s.sale_date
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ), 2) AS running_total
FROM company.sales s
JOIN company.products p
  ON s.product_id = p.product_id
ORDER BY s.sale_date, s.sale_id;
```

**Explanation:**

- `SUM(...) OVER (ORDER BY sale_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` creates a running total. For each row, it sums revenue from the very first row up to and including the current row.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` is the default behavior in PostgreSQL when you use `ORDER BY` in a window frame, but writing it explicitly is best practice for clarity and portability.
- `ROUND(..., 2)` keeps the currency values clean.

**Partitioned running total:** If you wanted a running total per region instead, you would add `PARTITION BY s.region`:
```sql
SUM(s.quantity * p.unit_price) OVER (
    PARTITION BY s.region
    ORDER BY s.sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) AS running_total_per_region
```

**Expected result:** The last row's `running_total` should equal the sum of all individual revenue values across the entire sales table.
</details>

---

## Problem 5: Employee Salary vs Department Average

**Business Question:**  
HR wants to compare each employee's salary to their department's average. For every employee, show their first name, last name, department name, their salary, the **average salary of their department**, and the **difference** between their salary and the department average.

Use the `AVG()` window function with `PARTITION BY` so that you don't collapse rows — each employee should still appear as one row. Order by department name, then by salary difference descending (so the highest-earners-above-average appear first).

In [ ]:
# Write your query here
query_5 = """

"""
run_query(query_5)

> **HINT** - scroll down only if stuck  
> Use `AVG(e.salary) OVER (PARTITION BY d.dept_name)` to compute the department average without collapsing rows. Then subtract: `e.salary - avg_dept_salary AS diff_from_avg`. Join to `departments` for the department name. Round the average to 2 decimal places for readability.

<details>
<summary><strong>SOLUTION - Problem 5</strong> (click to reveal)</summary>

```sql
SELECT e.first_name,
       e.last_name,
       d.dept_name,
       e.salary,
       ROUND(AVG(e.salary) OVER (PARTITION BY d.dept_name), 2) AS dept_avg_salary,
       ROUND(e.salary - AVG(e.salary) OVER (PARTITION BY d.dept_name), 2) AS diff_from_avg
FROM company.employees e
JOIN company.departments d
  ON e.department_id = d.dept_id
ORDER BY d.dept_name, diff_from_avg DESC;
```

**Explanation:**

- `AVG(e.salary) OVER (PARTITION BY d.dept_name)` computes the average salary per department. Unlike `GROUP BY`, this does **not** collapse rows — each employee row retains its individual identity while also carrying the department average.
- `diff_from_avg` shows whether an employee earns above or below their department average. Positive values mean above average; negative values mean below average.

**Why not GROUP BY?** With `GROUP BY d.dept_name`, you would get one row per department (the average), losing individual employee details. Window functions with `PARTITION BY` give you both: individual rows plus aggregated context.

**Other aggregates you can use this way:**
- `MIN(salary) OVER (PARTITION BY dept_name)` — department minimum
- `MAX(salary) OVER (PARTITION BY dept_name)` — department maximum
- `COUNT(*) OVER (PARTITION BY dept_name)` — department headcount
- `SUM(salary) OVER (PARTITION BY dept_name)` — total department salary cost
</details>

---

## Problem 6: Month-over-Month Sales Change (LAG + Aggregation)

**Business Question:**  
The executive team wants to track sales performance trends. Calculate the **total sales revenue** per month, then compute the **percentage change** from the previous month.

Show the month (as a `YYYY-MM` string), the `monthly_total`, the `previous_month_total` (using `LAG`), and the `pct_change` (percentage difference from the previous month). For the first month, `pct_change` should be NULL since there is no previous month to compare against.

Order by month ascending.

In [ ]:
# Write your query here
query_6 = """

"""
run_query(query_6)

> **HINT** - scroll down only if stuck  
> This requires two steps, so use a CTE. Step 1: In a CTE, aggregate sales by month — use `TO_CHAR(sale_date, 'YYYY-MM') AS month` and `SUM(s.quantity * p.unit_price) AS monthly_total` (joining sales to products). Step 2: In the outer query, use `LAG(monthly_total) OVER (ORDER BY month)` to get the previous month's total. Then calculate `ROUND((monthly_total - prev_total) / prev_total * 100, 2) AS pct_change`.

<details>
<summary><strong>SOLUTION - Problem 6</strong> (click to reveal)</summary>

```sql
WITH monthly_sales AS (
    SELECT TO_CHAR(s.sale_date, 'YYYY-MM') AS month,
           ROUND(SUM(s.quantity * p.unit_price), 2) AS monthly_total
    FROM company.sales s
    JOIN company.products p
      ON s.product_id = p.product_id
    GROUP BY TO_CHAR(s.sale_date, 'YYYY-MM')
)
SELECT month,
       monthly_total,
       LAG(monthly_total) OVER (ORDER BY month) AS prev_month_total,
       ROUND(
           (monthly_total - LAG(monthly_total) OVER (ORDER BY month))
           / LAG(monthly_total) OVER (ORDER BY month) * 100,
           2
       ) AS pct_change
FROM monthly_sales
ORDER BY month;
```

**Explanation:**

1. **CTE (`monthly_sales`)**: Aggregates all sales revenue by month using `GROUP BY TO_CHAR(sale_date, 'YYYY-MM')`. This gives one row per month with the total revenue.
2. **Outer query**: Uses `LAG(monthly_total) OVER (ORDER BY month)` to look back one month. The percentage change is calculated as `(current - previous) / previous * 100`.

**Expected results:** Since the data spans January 2025 through December 2025, you should see 12 rows. The first row (2025-01) will have NULL for `prev_month_total` and `pct_change`. You should notice higher totals in Q4 (October-December) reflecting the end-of-year sales push.

**Cleaner version (avoiding repeated LAG calls):**

```sql
WITH monthly_sales AS (
    SELECT TO_CHAR(s.sale_date, 'YYYY-MM') AS month,
           ROUND(SUM(s.quantity * p.unit_price), 2) AS monthly_total
    FROM company.sales s
    JOIN company.products p
      ON s.product_id = p.product_id
    GROUP BY TO_CHAR(s.sale_date, 'YYYY-MM')
),
with_prev AS (
    SELECT month,
           monthly_total,
           LAG(monthly_total) OVER (ORDER BY month) AS prev_month_total
    FROM monthly_sales
)
SELECT month,
       monthly_total,
       prev_month_total,
       ROUND((monthly_total - prev_month_total) / prev_month_total * 100, 2) AS pct_change
FROM with_prev
ORDER BY month;
```

This version calls `LAG()` only once in the second CTE, making the final SELECT cleaner.

**Note on NULL handling:** Division by NULL (first month) produces NULL automatically, which is the correct behavior. If you wanted to show 0 instead, wrap with `COALESCE(pct_change, 0)`.
</details>

---

## Problem 7: Highest Earner Per Department with Department Average (CTEs + Window Functions)

**Business Question:**  
This is a complex query combining multiple techniques. For **each department**, find the employee with the **highest salary**. Show the department name, the employee's first name, last name, their salary, and the **average salary of that department**.

Use CTEs and window functions to solve this. Order by department name.

In [ ]:
# Write your query here
query_7 = """

"""
run_query(query_7)

> **HINT** - scroll down only if stuck  
> You need two window functions: `ROW_NUMBER() OVER (PARTITION BY dept_name ORDER BY salary DESC)` to identify the top earner per department, and `AVG(salary) OVER (PARTITION BY dept_name)` to compute the department average. Use a CTE to compute both, then in the outer query filter `WHERE rn = 1`. Join with `departments` to get the department name.

<details>
<summary><strong>SOLUTION - Problem 7</strong> (click to reveal)</summary>

```sql
WITH emp_ranked AS (
    SELECT e.first_name,
           e.last_name,
           d.dept_name,
           e.salary,
           ROW_NUMBER() OVER (PARTITION BY d.dept_name ORDER BY e.salary DESC) AS rn,
           ROUND(AVG(e.salary) OVER (PARTITION BY d.dept_name), 2) AS dept_avg_salary
    FROM company.employees e
    JOIN company.departments d
      ON e.department_id = d.dept_id
)
SELECT dept_name,
       first_name,
       last_name,
       salary,
       dept_avg_salary
FROM emp_ranked
WHERE rn = 1
ORDER BY dept_name;
```

**Explanation:** This query demonstrates the power of combining multiple window functions in a single CTE:

1. **`ROW_NUMBER()`**: Ranks employees within each department by salary descending. The highest earner gets `rn = 1`.
2. **`AVG() OVER (PARTITION BY ...)`**: Computes the department average salary, replicated on every employee row.
3. **Outer query**: Filters to `rn = 1`, keeping only the top earner per department, but still carrying the department average.

**Expected results (6 rows, one per department):**

| dept_name | first_name | last_name | salary | dept_avg_salary |
|-----------|-----------|-----------|--------|-----------------|
| Engineering | Alice | Chen | 145000 | ~94500 |
| Finance | Nathan | Harris | 135000 | ~86571 |
| Human Resources | Iris | Taylor | 115000 | ~79800 |
| Marketing | Uma | King | 120000 | ~83333 |
| Operations | James | Campbell | 125000 | ~82167 |
| Sales | Amy | Adams | 130000 | ~82889 |

**Why this beats a subquery approach:** Without window functions, you would need a correlated subquery for the department average and a separate subquery or self-join to find the max salary per department. Window functions let you compute both in a single scan.

**Tie note:** If two employees in the same department have the exact same highest salary, `ROW_NUMBER()` picks one arbitrarily. To show all top earners in case of ties, use `RANK()` instead of `ROW_NUMBER()`.
</details>

---

## Summary of Concepts Covered

| Problem | Key Concepts |
|---------|-------------|
| 1 | `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()` — comparing ranking functions with tied salaries |
| 2 | **Top-N per group** pattern — `ROW_NUMBER()` + CTE + `WHERE rn <= N` |
| 3 | `LAG()` for previous row value — computing date differences between consecutive events |
| 4 | Running total — `SUM() OVER (ORDER BY ... ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` |
| 5 | `AVG() OVER (PARTITION BY ...)` — per-group aggregate without collapsing rows |
| 6 | Month-over-month change — `LAG()` after `GROUP BY` aggregation, percentage change calculation |
| 7 | **Complex** — combining `ROW_NUMBER()` + `AVG() OVER (PARTITION BY ...)` in a single CTE |

**Key patterns to remember:**
- **Top-N per group**: `ROW_NUMBER() OVER (PARTITION BY group ORDER BY metric DESC)` in a CTE, then `WHERE rn <= N`.
- **Running totals**: `SUM(metric) OVER (ORDER BY time_column ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.
- **LAG/LEAD**: Access values from previous or subsequent rows without a self-join.
- **Window aggregates**: `AVG()`, `SUM()`, `MIN()`, `MAX()`, `COUNT()` with `OVER (PARTITION BY ...)` give you per-group aggregates while keeping individual rows.
- **Window functions cannot go in WHERE**: You must compute them in a CTE or subquery, then filter in an outer query.

**Next steps:** Try extending these queries — for example, calculate a 3-month moving average in Problem 6, or find the second-highest earner per department by changing `rn = 1` to `rn = 2` in Problem 7.